# Pipeline Step 3: Representation Learning and Fusion

## Inputs
- **EMA Pre-State Table**: `ema_pre_table.csv` (calculated player pre-match Exponential Moving Average form states from Notebook 2)
- **Joined and Encoded Dataset**: `final_total_input.xlsx` (merged shot event dataset containing one-hot encoded shot features and raw EMAs from Notebook 2)

## Outputs
- **Player Style Vectors**: `player_style_final.csv` (16-dimensional latent style vectors representing player form compressed from 37 EMA features)
- **Final Modeling Dataset (CSV)**: `latent_only_xg_input.csv` (final augmented shot-by-shot modeling dataset containing physical and style features)
- **Final Modeling Dataset (Excel)**: `latent_only_xg_input.xlsx` (final augmented shot-by-shot modeling dataset in Excel format)


In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import unicodedata

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Libraries loaded successfully. Device: {device}")

import random
random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)


Libraries loaded successfully. Device: cpu


### Configuration Parameters
Define the date splits for representation learning and chronological modeling split handling.

In [2]:
# Temporal cutoff for autoencoder scaling train split
TARGET_DATE = pd.to_datetime("2025-01-07")

# Temporal thresholds for modeling train-val-test splits
THRESHOLD_VAL = pd.to_datetime("2024-10-15")
THRESHOLD_TEST = pd.to_datetime("2025-01-07")

print(f"AE Target Split Date : {TARGET_DATE.date()}")
print(f"Modeling Train Threshold : {THRESHOLD_VAL.date()}")
print(f"Modeling Test Threshold  : {THRESHOLD_TEST.date()}")


AE Target Split Date : 2025-01-07
Modeling Train Threshold : 2024-10-15
Modeling Test Threshold  : 2025-01-07


### PyTorch Autoencoder Architecture
Define the `AggregationAutoencoder` class to compress the 37-dimensional player performance history down to a 16-dimensional latent style space.

In [3]:
class AggregationAutoencoder(nn.Module):
    def __init__(self, input_dim, encoding_dims=[128, 64, 32, 16], dropout_rate=0.2):
        super(AggregationAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, encoding_dims[0]),
            nn.BatchNorm1d(encoding_dims[0]),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(encoding_dims[0], encoding_dims[1]),
            nn.BatchNorm1d(encoding_dims[1]),
            nn.ReLU(),
            nn.Linear(encoding_dims[1], encoding_dims[2]),
            nn.ReLU(),
            nn.Linear(encoding_dims[2], encoding_dims[3]),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dims[3], encoding_dims[2]),
            nn.ReLU(),
            nn.Linear(encoding_dims[2], encoding_dims[1]),
            nn.ReLU(),
            nn.Linear(encoding_dims[1], encoding_dims[0]),
            nn.ReLU(),
            nn.Linear(encoding_dims[0], input_dim)
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def encode(self, x):
        return self.encoder(x)


### Autoencoder Dataset and Scaler Preparation
Load the preprocessed player EMA table, extract numerical columns to be scaled, and split the historical records into training and validation sets for model evaluation.

In [4]:
print("Loading player EMA pre-state table...")
ema_table = pd.read_csv("ema_pre_table.csv")
ema_table['Date'] = pd.to_datetime(ema_table['Date'])

# Exclude non-feature and metadata columns
exclude_cols = {'Player', 'Date', 'Minutes played', 'Team', 'Position', 'Score', 'Home/Away', 'Player_ID'}
ema_feature_list = [c for c in ema_table.columns if c not in exclude_cols and np.issubdtype(ema_table[c].dtype, np.number)]
print(f"Number of performance features for representation learning: {len(ema_feature_list)}")

# Train/validation split based on target date
history_df = ema_table[ema_table['Date'] < TARGET_DATE].copy()
train_df = history_df[history_df['Date'] < THRESHOLD_VAL].copy()
val_df   = history_df[history_df['Date'] >= THRESHOLD_VAL].copy()

# Scale features using StandardScaler (fit on train split only)
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_df[ema_feature_list])
val_scaled = scaler.transform(val_df[ema_feature_list])

class StyleDataset(Dataset):
    def __init__(self, data):
        self.data = torch.tensor(data, dtype=torch.float32)
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx], self.data[idx]

train_loader = DataLoader(StyleDataset(train_scaled), batch_size=32, shuffle=True, drop_last=True)
val_loader = DataLoader(StyleDataset(val_scaled), batch_size=32, shuffle=False)

print(f"Train dataloader: {len(train_loader)} batches | Val dataloader: {len(val_loader)} batches")


Loading player EMA pre-state table...
Number of performance features for representation learning: 37
Train dataloader: 289 batches | Val dataloader: 76 batches


### Autoencoder Training
Train the autoencoder for 100 epochs, saving the best parameters (lowest validation loss) to `best_style_model.pth`.

In [5]:
torch.manual_seed(42)
model = AggregationAutoencoder(input_dim=len(ema_feature_list)).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

best_loss = float('inf')
epochs = 100

print("Starting training on device...")
for epoch in range(epochs):
    model.train()
    train_loss = 0
    for x, _ in train_loader:
        x = x.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), x)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for bx, _ in val_loader:
            bx = bx.to(device)
            val_loss += criterion(model(bx), bx).item()
    avg_val_loss = val_loss / len(val_loader)
    
    new_best = False
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_style_model.pth')
        new_best = True
        
    if (epoch + 1) % 10 == 0 or epoch == 0:
        status = " (Saved Best Model *)" if new_best else ""
        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}{status}")

print(f"\nTraining complete! Best Validation Loss achieved: {best_loss:.6f}")


Starting training on device...
Epoch [1/100] | Train Loss: 0.637564 | Val Loss: 0.695197 (Saved Best Model *)
Epoch [10/100] | Train Loss: 0.231618 | Val Loss: 0.403282 (Saved Best Model *)
Epoch [20/100] | Train Loss: 0.172365 | Val Loss: 0.361812
Epoch [30/100] | Train Loss: 0.147268 | Val Loss: 0.311981
Epoch [40/100] | Train Loss: 0.129317 | Val Loss: 0.288249 (Saved Best Model *)
Epoch [50/100] | Train Loss: 0.120841 | Val Loss: 0.285838 (Saved Best Model *)
Epoch [60/100] | Train Loss: 0.114682 | Val Loss: 0.278199
Epoch [70/100] | Train Loss: 0.112314 | Val Loss: 0.269539
Epoch [80/100] | Train Loss: 0.099585 | Val Loss: 0.271124
Epoch [90/100] | Train Loss: 0.100092 | Val Loss: 0.280519
Epoch [100/100] | Train Loss: 0.094713 | Val Loss: 0.272115

Training complete! Best Validation Loss achieved: 0.242606


### Latent Style Features Extraction
Load the best model state, extract the 16-dimensional player style features for all records in the EMA table, and save them.

In [6]:
def extract_latent_features(full_ema, features, model_path, scaler_obj):
    m = AggregationAutoencoder(input_dim=len(features))
    m.load_state_dict(torch.load(model_path, map_location=device))
    m.eval()
    
    all_scaled = scaler_obj.transform(full_ema[features])
    with torch.no_grad():
        latent = m.encode(torch.tensor(all_scaled, dtype=torch.float32)).numpy()
        
    style_cols = [f'style_dim_{i}' for i in range(16)]
    return pd.concat([
        full_ema[['Player', 'Date']].reset_index(drop=True),
        pd.DataFrame(latent, columns=style_cols)
    ], axis=1)

player_style_final = extract_latent_features(ema_table, ema_feature_list, 'best_style_model.pth', scaler)
player_style_final.to_csv("player_style_final.csv", index=False)
print(f"Extracted and saved {len(player_style_final)} player style profiles to 'player_style_final.csv'.")


Extracted and saved 15409 player style profiles to 'player_style_final.csv'.


### Merging Style Features with Shot Events
Load shot events from `final_total_input.xlsx` and merge shooter and goalkeeper style representations.

In [7]:
def normalize_txt(s):
    if pd.isna(s): return ""
    s = str(s).lower().strip()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    return " ".join(s.split())

shots_df = pd.read_excel('final_total_input.xlsx')
shots_df['orig_id'] = np.arange(len(shots_df))
shots_df['date'] = pd.to_datetime(shots_df['date'])

style_df = pd.read_csv('player_style_final.csv')
style_df['Date'] = pd.to_datetime(style_df['Date'])

shots_df['shooter_norm'] = shots_df['shooter'].apply(normalize_txt)
shots_df['goalie_norm'] = shots_df['goalie'].apply(normalize_txt)
style_df['player_norm'] = style_df['Player'].apply(normalize_txt)

print(f"Loaded {len(shots_df)} shots and {len(style_df)} style vectors.")

# Merge shooter style
sh_style = style_df.copy().rename(columns={'player_norm': 'shooter_norm', 'Date': 'date'})
style_cols = [f'style_dim_{i}' for i in range(16)]
sh_style = sh_style[['shooter_norm', 'date'] + style_cols]
sh_style.columns = ['shooter_norm', 'date'] + [f'sh_style_{i}' for i in range(16)]
final_data = pd.merge(shots_df, sh_style, on=['shooter_norm', 'date'], how='left')

# Merge goalie style
gk_style = style_df.copy().rename(columns={'player_norm': 'goalie_norm', 'Date': 'date'})
gk_style = gk_style[['goalie_norm', 'date'] + style_cols]
gk_style.columns = ['goalie_norm', 'date'] + [f'gk_style_{i}' for i in range(16)]
final_data = pd.merge(final_data, gk_style, on=['goalie_norm', 'date'], how='left')

# Clean up raw EMA performance features and normalization columns
cols_to_remove = [c for c in final_data.columns if c.startswith('sh__pre__') or c.startswith('gk__pre__')]
final_data = final_data.drop(columns=cols_to_remove)
final_data = final_data.drop(columns=['shooter_norm', 'goalie_norm'])

# Fill missing style columns with 0
style_feature_cols = [c for c in final_data.columns if 'style_' in c]
final_data[style_feature_cols] = final_data[style_feature_cols].fillna(0.0)

print(f"Merged style features. Shape: {final_data.shape}")


Loaded 11571 shots and 15409 style vectors.
Merged style features. Shape: (11571, 88)


### Data Augmentation Functions
Define the functions to flip foot selection and shift shot coordinate locations.

In [8]:
def augment_flip_foot_onehot(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates a new foot-flipped row for each foot-based shot record,
    preserving chronological sorting with mergesort.
    """
    new_rows = []
    col_left = 'shots_enc__shots__ShotBodyPart_Left Foot'
    col_right = 'shots_enc__shots__ShotBodyPart_Right Foot'
    
    for i, row in df.iterrows():
        is_foot = (row[col_left] == 1) or (row[col_right] == 1)
        if is_foot:
            flipped = row.copy()
            flipped[col_left] = row[col_right]
            flipped[col_right] = row[col_left]
            new_rows.append((i + 0.1, flipped))
            
    df_aug = pd.concat(
        [df] + [pd.DataFrame([r[1]]) for r in sorted(new_rows, key=lambda x: x[0])]
    ).sort_index(kind="mergesort").reset_index(drop=True)
    return df_aug

def augment_shift_location_onehot(df: pd.DataFrame, shift_meters: float = 1.0) -> pd.DataFrame:
    """
    Creates a new row for each shot record with event coordinate positions
    shifted slightly by +/- 1 meter (clipped to [0, 100]), keeping chronological mergesort sorting.
    """
    new_rows = []
    np.random.seed(42)
    col_x = 'shots_enc__shots__EventX'
    col_y = 'shots_enc__shots__EventY'
    
    for i, row in df.iterrows():
        shifted = row.copy()
        for col in [col_x, col_y]:
            delta = np.random.choice([-shift_meters, shift_meters])
            shifted[col] = np.clip(row[col] + delta, 0, 100)
        new_rows.append((i + 0.2, shifted))
        
    df_aug = pd.concat(
        [df] + [pd.DataFrame([r[1]]) for r in sorted(new_rows, key=lambda x: x[0])]
    ).sort_index(kind="mergesort").reset_index(drop=True)
    return df_aug


### Training Split Augmentation Execution
Apply the chronological train-val-test split. Apply data augmentations strictly to the training split (`Date < 2024-10-15`) to prevent data leakage.

In [9]:
train_df = final_data[final_data['date'] < THRESHOLD_VAL].copy()
val_df = final_data[(final_data['date'] >= THRESHOLD_VAL) & (final_data['date'] < THRESHOLD_TEST)].copy()
test_df = final_data[final_data['date'] >= THRESHOLD_TEST].copy()

print(f"Splits before augmentation: Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}")

print("Applying foot flip and location shift augmentations to train split only...")
train_flipped = augment_flip_foot_onehot(train_df)
train_augmented = augment_shift_location_onehot(train_flipped)

print(f"Train size after augmentation: {len(train_augmented)}")

# Concatenate augmented train with raw val and test splits
final_df = pd.concat([train_augmented, val_df, test_df], ignore_index=True)
print(f"Final concatenated shape: {final_df.shape}")


Splits before augmentation: Train=6817 | Val=1866 | Test=2888
Applying foot flip and location shift augmentations to train split only...
Train size after augmentation: 24924
Final concatenated shape: (29678, 88)


### Column Reordering and Saving Output
Verify the column schema, reorder columns to match the target reference file, and save both `.csv` and `.xlsx` formats.

In [10]:
final_df.to_csv("latent_only_xg_input.csv", index=False)
final_df.to_excel("latent_only_xg_input.xlsx", index=False)
print(f"Saved final unified modeling datasets to 'latent_only_xg_input.csv' and 'latent_only_xg_input.xlsx'.")

# Also save the augmented version of final_total_input_augmented.xlsx to keep sizes in sync
print("Syncing and augmenting final_total_input_augmented.xlsx...")
df_ema_raw = pd.read_excel('final_total_input.xlsx')
df_ema_raw['orig_id'] = np.arange(len(df_ema_raw))
df_ema_raw['date'] = pd.to_datetime(df_ema_raw['date'])
train_ema = df_ema_raw[df_ema_raw['date'] < THRESHOLD_VAL].copy()
val_ema = df_ema_raw[(df_ema_raw['date'] >= THRESHOLD_VAL) & (df_ema_raw['date'] < THRESHOLD_TEST)].copy()
test_ema = df_ema_raw[df_ema_raw['date'] >= THRESHOLD_TEST].copy()

train_ema_flipped = augment_flip_foot_onehot(train_ema)
train_ema_augmented = augment_shift_location_onehot(train_ema_flipped)
final_ema_df = pd.concat([train_ema_augmented, val_ema, test_ema], ignore_index=True)
final_ema_df.to_excel('final_total_input_augmented.xlsx', index=False)
print(f"Augmented final_total_input_augmented.xlsx saved. Shape: {final_ema_df.shape}")


Saved final unified modeling datasets to 'latent_only_xg_input.csv' and 'latent_only_xg_input.xlsx'.
Syncing and augmenting final_total_input_augmented.xlsx...
Augmented final_total_input_augmented.xlsx saved. Shape: (29678, 130)
